# Wi-Fi Fingerprinting – Predicting Location from Wi-Fi Signals
*(Corrected version of the project, following the teacher's feedback)*

**Goal:** predict where a device is (floor **and** X/Y position) using **only Wi-Fi signal strengths (RSSI)**.

### What was fixed
| Problem in the old version | Fix in this version |
|---|---|
| Missing imports / undefined variables (`plt`, `jit`, `pcs`) crashed the notebook | Everything is imported and defined, cells run top-to-bottom |
| Reads the wrong CSV after upload (`mydata (4).csv`) | The notebook reads the file that was just uploaded |
| SGD model diverged (MAE = 100,095) | Fixed learning settings, and floor is treated as a *class* |
| Floor predicted with regression (e.g. 2.5) | Floor is **classification** (accuracy + confusion matrix) |
| Only floor was predicted | Floor **and** X/Y position are predicted |
| Train/test split not checked | Split **by measurement location** + automatic leakage check |
| Optimistic test (grid neighbours) | Extra **spatial hold-out** robustness test |
| Neighbouring locations with **identical** Wi-Fi data (48 pairs) could sit in train and test | **Twin-safe split**: identical fingerprints are always kept on the same side |
| No real **3D multi-floor** estimate (floors were stacked in one 2-D plane) | **3D system**: floor + position *inside* the floor plan, 3D error, 3D plots, error CDFs, per-floor analysis |
| K-Means k fixed, no analysis, wrong plot | Elbow + silhouette, clusters plotted, compared with floors |
| No conclusions | Conclusions at the end, computed from the results |

**How to run:** click **Runtime → Run all**, then choose `mydata.csv` in the upload cell.

In [ ]:
# ---------------------------------------------------------------
# Imports and global settings
# ---------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.metrics import (accuracy_score, confusion_matrix, silhouette_score,
                             adjusted_rand_score, mean_squared_error, r2_score)

RANDOM_STATE = 42      # same random numbers every run -> reproducible results
MISSING_RSSI = -100    # signal value used when an access point is not heard
sns.set_theme(style="whitegrid")

results = {}           # we store key numbers here and use them in the conclusions

## Step 0 – Upload `mydata.csv`

In [ ]:
# In Google Colab this opens a file chooser. Outside Colab it simply uses "mydata.csv".
try:
    from google.colab import files
    uploaded = files.upload()
    CSV_NAME = list(uploaded.keys())[-1]   # the file that was JUST uploaded (fixes the "mydata (4).csv" problem)
except ImportError:
    CSV_NAME = "mydata.csv"
print("Using file:", CSV_NAME)

## Step 1 – Load and explore the data
Each row of the file is **one access point (BSSID) heard at one measurement location**, with its signal strength (RSSI in dBm; closer to 0 = stronger).

In [ ]:
df = pd.read_csv(CSV_NAME)

print(df.head())
print()
df.info()
print()
print(df.describe())

# Important observation: the three floor plans are stored stacked in ONE coordinate system.
# Each floor occupies its own band of PosY, so PosY alone reveals the floor.
# This is why using PosX/PosY as inputs gave "almost perfect" results in the old version.
print("\nCoordinate range per floor:")
print(df.groupby("PosZ")[["PosX", "PosY"]].agg(["min", "max"]))

## Step 2 – Build one Wi-Fi fingerprint per measurement location
Every location was scanned many times (~1,600 rows per location). We:
1. average the RSSI of each access point at each location,
2. put each location in **one row** and each BSSID in **one column**,
3. fill access points that were not heard with a weak signal (-100 dBm).

Only these RSSI columns are used as **inputs**. PosX, PosY and PosZ are only **targets**.

In [ ]:
location_cols = ["PosX", "PosY", "PosZ"]

fingerprints = (df.groupby(location_cols + ["BSSID"])["RSSI"].mean()   # average repeated scans
                  .unstack("BSSID")                                      # one column per BSSID
                  .fillna(MISSING_RSSI)                                  # AP not heard -> -100
                  .reset_index())

X       = fingerprints.drop(columns=location_cols)      # INPUTS : Wi-Fi signals only
y_floor = fingerprints["PosZ"].values                   # TARGET 1: floor (2, 3, 4)
# The three floor plans are stored STACKED in one coordinate system (each floor has its own band of PosY).
# For real 3D positioning we need the position INSIDE the floor plan ("local" coordinates):
#     local X = PosX        local Y = PosY - start of that floor's band
global_xy      = fingerprints[["PosX", "PosY"]].values            # position in the stacked file coordinates
floor_origin_y = fingerprints.groupby("PosZ")["PosY"].min()       # where each floor's band starts
local_xy = np.column_stack([fingerprints["PosX"].values,
                            fingerprints["PosY"].values - fingerprints["PosZ"].map(floor_origin_y).values])
y_xy = local_xy                                         # TARGET 2: position inside the floor plan

# The file gives the floors no real height. As a default we use the vertical spacing of the stacked floor plans.
# >>> If you know the real floor height in the SAME unit as PosX/PosY, set it here. <<<
FLOOR_HEIGHT = float(np.diff(np.sort(floor_origin_y.values)).mean())
floors  = sorted(np.unique(y_floor))

print("Measurement locations:", len(fingerprints))
print("Local coordinate range per floor (floor plans have the same size):")
print(pd.DataFrame(local_xy, columns=["local X", "local Y"]).groupby(y_floor).agg(["min", "max"]))
print(f"Floor spacing used for the 3D error: {FLOOR_HEIGHT:.1f} units")
print("Wi-Fi features (BSSIDs):", X.shape[1])
print(fingerprints.groupby("PosZ").size().rename("locations per floor"))

# ---- Data-quality check: do different locations have IDENTICAL Wi-Fi fingerprints? ----
# Give every distinct fingerprint an id. Locations that share an id are "twins".
twin_group = pd.factorize(pd.util.hash_pandas_object(X, index=False))[0]
n_unique = len(np.unique(twin_group))
n_twin_groups = int((pd.Series(twin_group).value_counts() > 1).sum())
print(f"\nDistinct fingerprints: {n_unique} of {len(X)} locations -> {n_twin_groups} groups of locations share EXACTLY the same Wi-Fi data")
# Twins must be on the same floor and close together (this is a property of the dataset)
assert all(len(set(y_floor[twin_group == g])) == 1 for g in np.unique(twin_group)), "twins on different floors?"

# ---- Plot: where were the measurements taken? ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc = axes[0].scatter(fingerprints["PosX"], fingerprints["PosY"], c=fingerprints["PosZ"],
                     cmap="viridis", s=40, edgecolor="k", linewidth=0.4)
fig.colorbar(sc, ax=axes[0], ticks=floors, label="PosZ (floor)")
axes[0].set_title("All measurement locations coloured by floor")
for fl, col in zip(floors, ["tab:blue", "tab:orange", "tab:green"]):
    d = fingerprints[fingerprints["PosZ"] == fl]
    axes[1].scatter(d["PosX"], d["PosY"], s=40, c=col, label=f"Floor {fl} (n={len(d)})")
axes[1].set_title("Locations per floor"); axes[1].legend()
for ax in axes:
    ax.set_xlabel("PosX"); ax.set_ylabel("PosY")
plt.tight_layout(); plt.savefig("01_locations_map.png", dpi=150); plt.show()

## Step 3 – Train/test split that SEPARATES measurement locations (and their twins)
Each row of `fingerprints` is a **unique location**, so no location can be in both sets
(the old version split the raw readings, so the same location appeared in train *and* test).

Step 2 also showed that some **neighbouring locations have exactly the same Wi-Fi data** (twins). If one twin is in the training set
and the other in the test set, the model can simply *copy* the twin's answer. We therefore split by **twin group**: twins always go together.
We take about 20 % of each floor as the test set and then **check** there is no overlap.

In [ ]:
def twin_safe_split(seed, test_fraction=0.2):
    # Split the locations into train/test so that
    #  - every floor is represented in both sets,
    #  - locations with identical fingerprints (twins) are never separated.
    rng = np.random.RandomState(seed)
    test_groups = []
    for floor in floors:
        groups_on_floor = np.unique(twin_group[y_floor == floor])
        n_test = int(round(test_fraction * len(groups_on_floor)))
        test_groups += list(rng.choice(groups_on_floor, size=n_test, replace=False))
    is_test = np.isin(twin_group, test_groups)
    return np.where(~is_test)[0], np.where(is_test)[0]

train_idx, test_idx = twin_safe_split(RANDOM_STATE)

X_train, X_test   = X.iloc[train_idx], X.iloc[test_idx]
yf_train, yf_test = y_floor[train_idx], y_floor[test_idx]
xy_train, xy_test = y_xy[train_idx],   y_xy[test_idx]

# ---- Leakage checks ----
train_locs = set(map(tuple, fingerprints.loc[train_idx, location_cols].values))
test_locs  = set(map(tuple, fingerprints.loc[test_idx,  location_cols].values))
assert train_locs.isdisjoint(test_locs), "Leakage: a location is in both train and test!"
assert set(twin_group[train_idx]).isdisjoint(twin_group[test_idx]), "Leakage: identical fingerprint in train and test!"
print(f"Train locations: {len(train_idx)} | Test locations: {len(test_idx)}")
print("Locations shared between train and test:", len(train_locs & test_locs))
print("Identical fingerprints shared between train and test:", len(set(twin_group[train_idx]) & set(twin_group[test_idx])))

## Step 4 – Task A: predict the FLOOR from Wi-Fi signals
The floor (2, 3 or 4) is a **category**, so this is **classification** (a prediction like 2.5 makes no sense).
We compare two models from the original work:
* **SGD linear model** – a linear classifier trained with stochastic gradient descent (fixed so it no longer diverges),
* **Random Forest**,

and a **baseline** that always predicts the most common floor, so we can see whether the models really learn something.

In [ ]:
floor_models = {
    "SGD linear model": Pipeline([
        ("scaler", StandardScaler()),
        ("sgd", SGDClassifier(loss="log_loss", alpha=1e-3, max_iter=2000, tol=1e-4,
                              random_state=RANDOM_STATE))]),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
}

# Baseline: always guess the most frequent floor of the training set
majority = pd.Series(yf_train).mode()[0]
baseline_acc = accuracy_score(yf_test, np.full(len(yf_test), majority))
print(f"Baseline (always predict floor {majority}) test accuracy: {baseline_acc:.3f}\n")

floor_preds, floor_acc = {}, {}
for name, model in floor_models.items():
    model.fit(X_train, yf_train)
    floor_preds[name] = model.predict(X_test)
    train_acc = accuracy_score(yf_train, model.predict(X_train))
    floor_acc[name] = accuracy_score(yf_test, floor_preds[name])
    print(f"{name:18s} | train accuracy: {train_acc:.3f} | TEST accuracy (unseen locations): {floor_acc[name]:.3f}")

results["floor_acc"] = floor_acc
results["floor_baseline"] = baseline_acc

# ---- Plot: confusion matrix of each model ----
fig, axes = plt.subplots(1, len(floor_models), figsize=(11, 4))
for ax, (name, pred) in zip(axes, floor_preds.items()):
    sns.heatmap(confusion_matrix(yf_test, pred), annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=floors, yticklabels=floors, ax=ax)
    ax.set_title(f"{name}\n(test accuracy {floor_acc[name]:.2f})")
    ax.set_xlabel("Predicted floor"); ax.set_ylabel("True floor")
plt.tight_layout(); plt.savefig("02_floor_confusion_matrices.png", dpi=150); plt.show()

# ---- Plot: where on the floor plan are the test locations predicted correctly? ----
best_floor_model = max(floor_acc, key=floor_acc.get)
ok = floor_preds[best_floor_model] == yf_test
plt.figure(figsize=(8, 7))
gxy_train, gxy_test = global_xy[train_idx], global_xy[test_idx]      # stacked file coordinates (shows all floors)
plt.scatter(gxy_train[:, 0], gxy_train[:, 1], s=10, c="lightgrey", label="Train locations")
plt.scatter(gxy_test[ok, 0],  gxy_test[ok, 1],  s=45, c="green", label="Test: floor correct")
plt.scatter(gxy_test[~ok, 0], gxy_test[~ok, 1], s=70, c="red", marker="x", label="Test: floor wrong")
plt.title(f"Floor prediction on unseen locations - {best_floor_model}")
plt.xlabel("PosX"); plt.ylabel("PosY"); plt.legend()
plt.tight_layout(); plt.savefig("03_floor_prediction_map.png", dpi=150); plt.show()

## Step 5 – Task B: predict the (X, Y) position inside the floor plan
Now we predict the position **inside the floor plan** (local X, local Y), again from Wi-Fi signals only and **without being told the floor**.
The **error** is the straight-line distance between the true and predicted position (in the coordinate units of the file).
A "mean position" baseline shows what no skill looks like.
> The polynomial model first compresses the 600+ signals with PCA (20 components) – using degree-2 polynomial features directly on all BSSIDs would create ~195,000 columns from only a few hundred locations.

In [ ]:
def position_error(pred, true):
    # Straight-line distance between predicted and true position for every location
    return np.sqrt(((pred - true) ** 2).sum(axis=1))

xy_models = {
    "KNN (k=3)": Pipeline([("scaler", StandardScaler()),
                           ("knn", KNeighborsRegressor(n_neighbors=3, weights="distance"))]),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "Polynomial Ridge (PCA + degree 2)": Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=20, random_state=RANDOM_STATE)),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler2", StandardScaler()),
        ("ridge", Ridge(alpha=100))]),
}

# Baseline: always predict the average training position
base_err = position_error(np.tile(xy_train.mean(axis=0), (len(xy_test), 1)), xy_test)

xy_preds, xy_errors, rows = {}, {}, []
for name, model in xy_models.items():
    model.fit(X_train, xy_train)
    xy_preds[name] = model.predict(X_test)
    xy_errors[name] = position_error(xy_preds[name], xy_test)
    rows.append({"Model": name,
                 "R2": r2_score(xy_test, xy_preds[name]),
                 "Mean error": xy_errors[name].mean(),
                 "Median error": np.median(xy_errors[name])})
rows.append({"Model": "Baseline (mean position)", "R2": np.nan,
             "Mean error": base_err.mean(), "Median error": np.median(base_err)})
xy_table = pd.DataFrame(rows).set_index("Model").round(2)
print(xy_table)
results["xy_table"] = xy_table

# ---- Plot 1: mean / median error per model ----
xy_table[["Mean error", "Median error"]].plot(kind="bar", figsize=(8, 4))
plt.ylabel("Position error (coordinate units)"); plt.xticks(rotation=20, ha="right")
plt.title("Position error on unseen test locations (lower is better)")
plt.tight_layout(); plt.savefig("04_position_error_by_model.png", dpi=150); plt.show()

# ---- Plot 2: true vs predicted position for the best model ----
best_xy_model = min(xy_errors, key=lambda k: xy_errors[k].mean())
pred = xy_preds[best_xy_model]
plt.figure(figsize=(8, 7))
plt.scatter(xy_train[:, 0], xy_train[:, 1], s=10, c="lightgrey", label="Train locations")
plt.scatter(xy_test[:, 0], xy_test[:, 1], s=35, c="green", label="True test location")
plt.scatter(pred[:, 0], pred[:, 1], s=35, c="red", marker="x", label="Predicted")
for t, p in zip(xy_test, pred):
    plt.plot([t[0], p[0]], [t[1], p[1]], c="red", alpha=0.3, lw=0.8)   # line = prediction error
plt.title(f"Predicted vs true position (all floors overlaid) - {best_xy_model}")
plt.xlabel("local X"); plt.ylabel("local Y"); plt.legend()
plt.tight_layout(); plt.savefig("05_position_predictions_map.png", dpi=150); plt.show()

# ---- Plot 3: distribution of the errors ----
plt.figure(figsize=(8, 4))
plt.hist(xy_errors[best_xy_model], bins=20, color="steelblue", edgecolor="k")
plt.axvline(np.median(xy_errors[best_xy_model]), c="r", ls="--", label="median")
plt.xlabel("Position error"); plt.ylabel("Number of test locations")
plt.title(f"Error distribution - {best_xy_model}"); plt.legend()
plt.tight_layout(); plt.savefig("06_position_error_histogram.png", dpi=150); plt.show()

## Step 6 – The final system: 3D multi-floor indoor positioning
A 3D position has **three parts**: the **floor (Z)** and the **position inside the floor plan (X, Y)**.
The final estimator therefore works in two stages:
1. **Floor detection** – classify the floor from the Wi-Fi signals (SGD classifier).
2. **Position inside the floor** – a separate regressor per floor, trained only with that floor's locations (KNN).

Error measures (standard for multi-floor positioning):
* **Floor accuracy** – how often the correct floor is found.
* **2-D error** – straight-line distance inside the floor plan (local X, Y).
* **3-D error** – combines the 2-D error with the height difference when the floor is wrong: `sqrt(2-D error² + (floor difference × FLOOR_HEIGHT)²)`.

We first choose the number of neighbours **k** with cross-validation **on the training set only**, then compare three configurations on the unseen test locations.
The *oracle* configuration is given the true floor; it shows the best possible position error if floor detection were perfect.

In [ ]:
def make_sgd():
    # Floor classifier (same settings as in Step 4)
    return Pipeline([("scaler", StandardScaler()),
                     ("sgd", SGDClassifier(loss="log_loss", alpha=1e-3, max_iter=2000, tol=1e-4,
                                           random_state=RANDOM_STATE))])

def make_knn(k=3):
    # Position regressor: average of the k most similar fingerprints (closer ones count more)
    return Pipeline([("scaler", StandardScaler()),
                     ("knn", KNeighborsRegressor(n_neighbors=k, weights="distance"))])

def predict_3d(tr, te, floor_model=None, make_regressor=make_knn):
    # Predict floor and position inside the floor for the locations `te`, learning only from `tr`.
    # floor_model=None means "oracle": the true floor is used.
    if floor_model is None:
        floor_pred = y_floor[te]
    else:
        floor_pred = floor_model.fit(X.iloc[tr], y_floor[tr]).predict(X.iloc[te])
    xy_pred = np.zeros((len(te), 2))
    for f in floors:                                   # stage 2: one regressor per (predicted) floor
        mask = floor_pred == f
        if mask.sum() == 0:
            continue
        tr_f = tr[y_floor[tr] == f]
        xy_pred[mask] = make_regressor().fit(X.iloc[tr_f], local_xy[tr_f]).predict(X.iloc[te][mask])
    return floor_pred, xy_pred

def errors_3d(te, floor_pred, xy_pred):
    # Per-location errors
    err_2d = np.sqrt(((xy_pred - local_xy[te]) ** 2).sum(axis=1))
    err_3d = np.sqrt(err_2d ** 2 + ((floor_pred - y_floor[te]) * FLOOR_HEIGHT) ** 2)
    return err_2d, err_3d

def summarize_3d(te, floor_pred, xy_pred):
    err_2d, err_3d = errors_3d(te, floor_pred, xy_pred)
    right = floor_pred == y_floor[te]
    return {"Floor accuracy": right.mean(),
            "2D error mean": err_2d.mean(), "2D error median": np.median(err_2d),
            "2D error 90th pct": np.percentile(err_2d, 90),
            "2D error (correct floor only)": err_2d[right].mean(),
            "3D error mean": err_3d.mean(), "3D error median": np.median(err_3d)}

# ---- 1) choose k with cross-validation on the TRAINING locations only (twin-safe) ----
inner = StratifiedGroupKFold(5, shuffle=True, random_state=RANDOM_STATE)
k_values = list(range(1, 11))
k_scores = []
for k in k_values:
    fold_err = []
    for a, b in inner.split(train_idx, y_floor[train_idx], twin_group[train_idx]):
        tr_in, va_in = train_idx[a], train_idx[b]
        fp_, xp_ = predict_3d(tr_in, va_in, make_sgd(), lambda: make_knn(k))
        fold_err.append(errors_3d(va_in, fp_, xp_)[0].mean())
    k_scores.append(np.mean(fold_err))
best_k = k_values[int(np.argmin(k_scores))]
print("Cross-validated 2-D error for k = 1..10:", np.round(k_scores, 1))
print("Chosen k:", best_k)

plt.figure(figsize=(7, 4))
plt.plot(k_values, k_scores, "o-"); plt.axvline(best_k, c="r", ls="--", label=f"chosen k = {best_k}")
plt.xlabel("k (number of neighbours)"); plt.ylabel("2-D error (cross-validation, training set)")
plt.title("Choosing k without touching the test set"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("07_choose_k.png", dpi=150); plt.show()

# ---- 2) compare three 3D configurations on the unseen test locations ----
configs = {
    f"SGD floor -> KNN position (k={best_k})": (make_sgd(), lambda: make_knn(best_k)),
    "Random Forest floor -> Random Forest position": (
        RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        lambda: RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
    f"Oracle (true floor) -> KNN position (k={best_k})": (None, lambda: make_knn(best_k)),
}
final_preds, final_rows = {}, {}
for name, (floor_model, make_reg) in configs.items():
    fp_, xp_ = predict_3d(train_idx, test_idx, floor_model, make_reg)
    final_preds[name] = (fp_, xp_)
    final_rows[name] = summarize_3d(test_idx, fp_, xp_)
final_table = pd.DataFrame(final_rows).T.round(3)
print("\n3D positioning results on UNSEEN test locations:")
print(final_table.to_string())

main_name = list(configs)[0]                         # the main system of this project
results["final_table"], results["main_name"] = final_table, main_name
results["final_rows"] = final_rows

In [ ]:
# ---------------- Plots of the 3D system (main configuration) ----------------
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401  (needed for 3-D plots)

fp_main, xp_main = final_preds[main_name]
true_xy, true_z = local_xy[test_idx], y_floor[test_idx]
err2d_main, err3d_main = errors_3d(test_idx, fp_main, xp_main)

# ---- Plot 1: true vs predicted positions in 3D ----
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(local_xy[train_idx, 0], local_xy[train_idx, 1], y_floor[train_idx],
           c="lightgrey", s=8, alpha=0.5, label="Train locations")
ax.scatter(true_xy[:, 0], true_xy[:, 1], true_z, c="green", s=35, label="True test position")
ax.scatter(xp_main[:, 0], xp_main[:, 1], fp_main, c="red", marker="x", s=40, label="Predicted position")
for t, p, tz, pz in zip(true_xy, xp_main, true_z, fp_main):
    ax.plot([t[0], p[0]], [t[1], p[1]], [tz, pz], c="red", alpha=0.35, lw=0.8)
ax.set_xlabel("local X"); ax.set_ylabel("local Y"); ax.set_zlabel("Floor")
ax.set_zticks(floors); ax.set_box_aspect((4, 1.2, 2)); ax.view_init(elev=22, azim=-62)
ax.set_title(f"3D positioning on unseen locations\n{main_name}"); ax.legend(loc="upper left")
plt.tight_layout(); plt.savefig("08_3d_positions.png", dpi=150); plt.show()

# ---- Plot 2: the same result floor by floor (easier to read) ----
fig, axes = plt.subplots(len(floors), 1, figsize=(13, 3.2 * len(floors)), sharex=True)
for ax, f in zip(axes, floors):
    on_floor = true_z == f
    ax.scatter(local_xy[train_idx][y_floor[train_idx] == f][:, 0],
               local_xy[train_idx][y_floor[train_idx] == f][:, 1], s=8, c="lightgrey")
    ax.scatter(true_xy[on_floor, 0], true_xy[on_floor, 1], s=35, c="green", label="True")
    ax.scatter(xp_main[on_floor, 0], xp_main[on_floor, 1], s=35, c="red", marker="x", label="Predicted")
    for t, p in zip(true_xy[on_floor], xp_main[on_floor]):
        ax.plot([t[0], p[0]], [t[1], p[1]], c="red", alpha=0.4, lw=0.9)
    wrong = (on_floor & (fp_main != true_z)).sum()
    ax.set_title(f"Floor {f}: {on_floor.sum()} test locations, floor wrong for {wrong}")
    ax.set_ylabel("local Y")
axes[0].legend(ncol=2); axes[-1].set_xlabel("local X")
plt.tight_layout(); plt.savefig("09_positions_per_floor.png", dpi=150); plt.show()

# ---- Plot 3: cumulative distribution of the error (standard in indoor positioning) ----
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for name in configs:
    e2, e3 = errors_3d(test_idx, *final_preds[name])
    for a, e in zip(ax, (e2, e3)):
        a.plot(np.sort(e), np.arange(1, len(e) + 1) / len(e), lw=2, label=name)
ax[0].set_xlabel("2-D error (inside the floor plan)"); ax[1].set_xlabel("3-D error")
for a in ax:
    a.set_ylabel("Fraction of test locations with error <= x"); a.grid(alpha=0.3)
ax[0].set_title("Error CDF - 2D"); ax[1].set_title("Error CDF - 3D"); ax[0].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.savefig("10_error_cdf.png", dpi=150); plt.show()

# ---- Plot 4: results per floor ----
per_floor = pd.DataFrame({
    "Floor accuracy": [(fp_main[true_z == f] == f).mean() for f in floors],
    "2D error mean": [err2d_main[true_z == f].mean() for f in floors],
    "2D error median": [np.median(err2d_main[true_z == f]) for f in floors]},
    index=[f"Floor {f}" for f in floors]).round(3)
print(per_floor)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
per_floor["Floor accuracy"].plot(kind="bar", ax=ax[0], color="tab:green", edgecolor="k"); ax[0].set_ylim(0, 1.05)
ax[0].set_title("Floor accuracy per floor"); ax[0].tick_params(axis="x", rotation=0)
per_floor[["2D error mean", "2D error median"]].plot(kind="bar", ax=ax[1], edgecolor="k")
ax[1].set_title("2-D position error per floor"); ax[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.savefig("11_results_per_floor.png", dpi=150); plt.show()

## Step 7 – Robustness check: how strict is our test?
The measurement locations lie on a regular grid (~20 units apart), so even a location-based random split puts each test point **next to** a training point. This is optimistic.

A **stricter** test cuts the map into vertical strips (`PosX // 400`) and holds out **whole strips on all floors** together.
We compare both tests with 5-fold cross-validation, using the **final 3D system of Step 6** (floor first, then position inside that floor).

In [ ]:
def cross_validate(splits):
    # Run the FINAL 3D system (SGD floor -> KNN position) on every fold and average the results
    rows = []
    for tr, te in splits:
        fp_, xp_ = predict_3d(tr, te, make_sgd(), lambda: make_knn(best_k))
        rows.append(summarize_3d(te, fp_, xp_))
    return pd.DataFrame(rows).mean().to_dict()

# Test A: random locations, 5-fold, twins kept together
random_cv = cross_validate(list(StratifiedGroupKFold(5, shuffle=True, random_state=RANDOM_STATE)
                                .split(X, y_floor, twin_group)))
# Test B: whole vertical strips of the map are held out on ALL floors (twins stay together)
group_min_x = pd.Series(fingerprints["PosX"].values).groupby(twin_group).transform("min").values
strips = group_min_x // 400
spatial_cv = cross_validate(list(GroupKFold(5).split(X, y_floor, strips)))

keys = ["Floor accuracy", "2D error mean", "2D error median", "2D error 90th pct", "3D error mean"]
robust = pd.DataFrame({"5-fold, random locations (twin-safe)": {k: random_cv[k] for k in keys},
                       "5-fold, spatial strips held out": {k: spatial_cv[k] for k in keys}}).round(3)
print(robust)
results["robust"] = robust

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
robust.loc[["Floor accuracy"]].T.plot(kind="bar", ax=ax[0], legend=False, color=["tab:blue", "tab:orange"], edgecolor="k")
ax[0].set_ylim(0, 1.05); ax[0].set_title("Floor accuracy"); ax[0].tick_params(axis="x", rotation=15)
robust.loc[["2D error mean", "2D error median", "3D error mean"]].plot(kind="bar", ax=ax[1], edgecolor="k")
ax[1].set_title("Position error (coordinate units)"); ax[1].tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.savefig("12_robustness.png", dpi=150); plt.show()

## Step 8 – Clustering of the raw Wi-Fi fingerprints (K-Means)
The old clustering used only coordinates. Now we cluster the **Wi-Fi signals**. Floors and coordinates are **not** used by the algorithm –
we look at them only afterwards to interpret the clusters.
(With only 273 rows, normal K-Means is preferable to Mini-Batch K-Means, which is meant for huge datasets.)

In [ ]:
# Scale the signals, then compress to 20 components (removes noise, makes clustering stable)
X_scaled = StandardScaler().fit_transform(X)
X_pca20  = PCA(n_components=20, random_state=RANDOM_STATE).fit_transform(X_scaled)

# Choose k with the elbow method and the silhouette score
ks = range(2, 11)
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_pca20)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X_pca20, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(ks), inertia, "o--"); ax[0].set_title("Elbow method"); ax[0].set_xlabel("k"); ax[0].set_ylabel("Inertia")
ax[1].plot(list(ks), sil, "o--", c="g"); ax[1].set_title("Silhouette score"); ax[1].set_xlabel("k"); ax[1].set_ylabel("Silhouette")
plt.tight_layout(); plt.savefig("13_kmeans_elbow_silhouette.png", dpi=150); plt.show()

# No clear elbow / silhouette peak, so we use k = 3 (= number of floors).
# This lets us test directly whether Wi-Fi fingerprints alone separate the floors.
raw_kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10).fit(X_pca20)
raw_labels = raw_kmeans.labels_
raw_ari = adjusted_rand_score(y_floor, raw_labels)      # 1 = clusters match floors, 0 = random
results["raw_ari"] = raw_ari

print("Clusters vs floor:")
print(pd.crosstab(pd.Series(raw_labels, name="Cluster"), pd.Series(y_floor, name="Floor")))
print(f"\nAdjusted Rand Index (raw fingerprints vs floor): {raw_ari:.3f}")

# ---- Plots: 2-D PCA view coloured by cluster and by true floor ----
pc2 = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].scatter(pc2[:, 0], pc2[:, 1], c=raw_labels, cmap="viridis", s=40)
ax[0].set_title("K-Means clusters (k=3) of raw Wi-Fi fingerprints")
sc = ax[1].scatter(pc2[:, 0], pc2[:, 1], c=y_floor, cmap="viridis", s=40)
fig.colorbar(sc, ax=ax[1], ticks=floors, label="True floor"); ax[1].set_title("Same points coloured by TRUE floor")
for a in ax:
    a.set_xlabel("PC 1"); a.set_ylabel("PC 2")
plt.tight_layout(); plt.savefig("14_raw_clusters_pca.png", dpi=150); plt.show()

# ---- Plot: clusters drawn on the floor plans ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
for ax, fl in zip(axes, floors):
    m = y_floor == fl
    ax.scatter(fingerprints.loc[m, "PosX"], fingerprints.loc[m, "PosY"], c=raw_labels[m],
               cmap="tab10", vmin=0, vmax=9, s=45)
    ax.set_title(f"Floor {fl}"); ax.set_xlabel("PosX")
axes[0].set_ylabel("PosY")
plt.suptitle("Raw-fingerprint clusters on each floor plan (colour = cluster)")
plt.tight_layout(); plt.savefig("15_raw_clusters_floorplan.png", dpi=150); plt.show()

# ---- Plot: floor composition of every cluster ----
pd.crosstab(raw_labels, y_floor).plot(kind="bar", stacked=True, figsize=(7, 5), edgecolor="k")
plt.xlabel("Cluster"); plt.ylabel("Number of locations"); plt.xticks(rotation=0)
plt.title("Floor composition of each raw-fingerprint cluster"); plt.legend(title="True floor")
plt.tight_layout(); plt.savefig("16_raw_clusters_composition.png", dpi=150); plt.show()

## Step 9 – Clustering by *BSSID groups per floor* and their signal strength
Raw fingerprints are dominated by **where on the floor** the device is, so clusters do not follow the floors.
A better idea: every access point is mounted on **one floor** and is heard strongest there.
1. Assign every BSSID to the floor where its **average** signal is strongest (computed from **training locations only**).
2. For every location, compute the average signal strength of the floor-2 BSSIDs, floor-3 BSSIDs and floor-4 BSSIDs.
3. Cluster these 3 numbers with K-Means (no labels are used in the clustering).

In [ ]:
# Step 1: which floor does each BSSID belong to? (training locations only -> test locations stay unseen)
ap_floor = X_train.groupby(yf_train).mean().idxmax()
print("BSSIDs assigned to each floor:")
print(ap_floor.value_counts().sort_index())

# Step 2: three features per location = mean signal of each floor's BSSID group
floor_signal = pd.DataFrame({f"Floor {f} BSSIDs": X.loc[:, ap_floor == f].mean(axis=1) for f in floors})
fs_scaled = StandardScaler().fit_transform(floor_signal)

# Step 3: K-Means with k = 3
fs_kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10).fit(fs_scaled)
fs_labels = fs_kmeans.labels_
fs_ari_all  = adjusted_rand_score(y_floor, fs_labels)
fs_ari_test = adjusted_rand_score(yf_test, fs_labels[test_idx])
results["fs_ari_all"], results["fs_ari_test"] = fs_ari_all, fs_ari_test
results["fs_silhouette"] = silhouette_score(fs_scaled, fs_labels)

print("\nCluster vs true floor:")
print(pd.crosstab(pd.Series(fs_labels, name="Cluster"), pd.Series(y_floor, name="Floor")))
print(f"\nAdjusted Rand Index - all locations : {fs_ari_all:.3f}")
print(f"Adjusted Rand Index - test locations: {fs_ari_test:.3f}")

# ---- Plot: clusters in 2-D (left) and the true floors (right) ----
pca_fs = PCA(n_components=2, random_state=RANDOM_STATE).fit(fs_scaled)
p2, cent = pca_fs.transform(fs_scaled), pca_fs.transform(fs_kmeans.cluster_centers_)
floor_colors = {2: "tab:blue", 3: "tab:orange", 4: "tab:green"}

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].scatter(p2[:, 0], p2[:, 1], c=fs_labels, cmap="viridis", s=35)
ax[0].scatter(cent[:, 0], cent[:, 1], c="red", marker="X", s=250, edgecolors="black", label="Centroids")
ax[0].set_title("K-Means clusters (k=3) on floor-wise BSSID signal strength"); ax[0].legend()
for f in floors:
    m = y_floor == f
    ax[1].scatter(p2[m, 0], p2[m, 1], c=floor_colors[f], s=35, label=f"Floor {f}")
ax[1].set_title("Same points coloured by TRUE floor"); ax[1].legend()
for a in ax:
    a.set_xlabel("Component 1"); a.set_ylabel("Component 2")
plt.tight_layout(); plt.savefig("17_floor_signal_clusters.png", dpi=150); plt.show()

# ---- Plot: signal strength of every BSSID group on every floor ----
avg = floor_signal.groupby(y_floor).mean()
avg.index = [f"Floor {f}" for f in avg.index]
plt.figure(figsize=(7, 4))
sns.heatmap(avg, annot=True, fmt=".1f", cmap="viridis_r", cbar_kws={"label": "mean RSSI (dBm)"})
plt.title("Average signal: BSSID group (columns) vs floor where measured (rows)")
plt.tight_layout(); plt.savefig("18_floor_signal_heatmap.png", dpi=150); plt.show()

# ---- Plot: pairs of floor signals (blob-style scatter coloured by true floor) ----
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
cols = list(floor_signal.columns)
for a, (i, j) in zip(ax, [(0, 1), (0, 2), (1, 2)]):
    for f in floors:
        m = y_floor == f
        a.scatter(floor_signal.iloc[m, i], floor_signal.iloc[m, j], c=floor_colors[f], s=30, label=f"Floor {f}")
    a.set_xlabel(cols[i] + " (mean RSSI)"); a.set_ylabel(cols[j] + " (mean RSSI)")
ax[0].legend(); plt.suptitle("Signal strength of each floor's BSSIDs, locations coloured by floor")
plt.tight_layout(); plt.savefig("19_floor_signal_pairs.png", dpi=150); plt.show()

# ---- Plot: floor composition of each cluster ----
pd.crosstab(fs_labels, y_floor).plot(kind="bar", stacked=True, figsize=(7, 5), edgecolor="k",
                                     color=[floor_colors[f] for f in floors])
plt.xlabel("Cluster"); plt.ylabel("Number of locations"); plt.xticks(rotation=0)
plt.title("Floor composition of each cluster (floor-wise BSSID signal)"); plt.legend(title="True floor")
plt.tight_layout(); plt.savefig("20_floor_signal_composition.png", dpi=150); plt.show()

## Step 10 – Is something leaking, or are the models just memorising?
The results are very accurate, so we test this directly. Each check below answers one specific doubt.

| Doubt | Test | What a "leak / memorisation" would look like |
|---|---|---|
| Do the inputs contain the answer? | List the input columns | PosX / PosY / PosZ among the features |
| Can a test location be copied from training? | Count identical fingerprints shared by train and test | A number above 0 |
| Is the model really learning from the signals? | **Shuffle the labels** and retrain | Still high accuracy |
| Was it just a lucky split? | Repeat the split 20 times | Large swings between splits |
| Does it only work because it saw many examples? | Train on **very few** locations | Accuracy only high with lots of data |
| Is the task simply easy? | A **no-ML rule**: "the floor whose access points are loudest" | The simple rule is also very accurate |

In [ ]:
print("1) Input columns used by the models:", X.shape[1], "BSSID signal columns")
print("   Any target column (PosX/PosY/PosZ) among the inputs?", any(c in location_cols for c in X.columns))

print("\n2) Identical fingerprints shared by train and test:", len(set(twin_group[train_idx]) & set(twin_group[test_idx])))

# 3) Shuffled-label test: if the model had access to the answer some other way, it would still score high.
#    With shuffled labels a correct pipeline must fall back to chance (about 33 % for 3 floors).
# (make_sgd() is the floor classifier defined in Step 6)
rs = np.random.RandomState(0)
shuffled_acc = []
for _ in range(30):
    y_shuffled = rs.permutation(yf_train)
    shuffled_acc.append((make_sgd().fit(X_train, y_shuffled).predict(X_test) == yf_test).mean())
print(f"\n3) Floor accuracy with SHUFFLED training labels (30 runs): mean {np.mean(shuffled_acc):.3f} "
      f"(chance = {1/len(floors):.3f}), best single run {np.max(shuffled_acc):.3f}")

# 4) Repeat the twin-safe split with 20 different random seeds
rep_acc = []
for seed in range(20):
    tr, te = twin_safe_split(seed)
    rep_acc.append((make_sgd().fit(X.iloc[tr], y_floor[tr]).predict(X.iloc[te]) == y_floor[te]).mean())
print(f"\n4) Floor accuracy over 20 different twin-safe splits: mean {np.mean(rep_acc):.3f}, worst {np.min(rep_acc):.3f}")

# 5) Learning with very little data (memorising needs many examples; learning does not)
print("\n5) Floor accuracy when training on only a few locations per floor (rest = test):")
small_train_acc = {}
for n in [3, 6, 15, 30]:
    accs = []
    for seed in range(10):
        rs = np.random.RandomState(seed)
        tr = np.concatenate([rs.choice(np.where(y_floor == f)[0], n, replace=False) for f in floors])
        te = np.setdiff1d(np.arange(len(X)), tr)
        te = te[~np.isin(twin_group[te], twin_group[tr])]          # keep it twin-safe
        accs.append((make_sgd().fit(X.iloc[tr], y_floor[tr]).predict(X.iloc[te]) == y_floor[te]).mean())
    small_train_acc[n * len(floors)] = np.mean(accs)
    print(f"   {n * len(floors):3d} training locations -> accuracy {np.mean(accs):.3f}")

# 6) Is the task simply easy? A rule WITHOUT any machine learning:
#    "the floor whose access points are heard loudest on average"
ap_floor_rule = X_train.groupby(yf_train).mean().idxmax()            # BSSID -> floor (training data only)
signal_per_floor = np.column_stack([X_test.loc[:, ap_floor_rule == f].mean(axis=1) for f in floors])
rule_acc = (np.array(floors)[signal_per_floor.argmax(axis=1)] == yf_test).mean()
print(f"\n6) No-ML rule 'floor whose access points are loudest': test accuracy {rule_acc:.3f}")

results["shuffled_acc"], results["repeat_acc"] = np.mean(shuffled_acc), rep_acc
results["small_train_acc"], results["rule_acc"] = small_train_acc, rule_acc

# ---- Plot: all sanity checks in one picture ----
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
labels = ["Shuffled\nlabels", "No-ML rule", "SGD model\n(normal)"]
vals = [np.mean(shuffled_acc), rule_acc, results["floor_acc"]["SGD linear model"]]
ax[0].bar(labels, vals, color=["tab:red", "tab:orange", "tab:green"], edgecolor="k")
ax[0].axhline(1 / len(floors), c="k", ls="--", label="chance"); ax[0].set_ylim(0, 1.05)
ax[0].set_ylabel("Floor accuracy (test)"); ax[0].set_title("Shuffled labels fall to chance; the signal is real"); ax[0].legend()
ax[1].plot(list(small_train_acc.keys()), list(small_train_acc.values()), "o-")
ax[1].set_xlabel("Number of TRAINING locations"); ax[1].set_ylabel("Floor accuracy (test)")
ax[1].set_title("Accuracy vs amount of training data"); ax[1].set_ylim(0, 1.05); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("21_sanity_checks.png", dpi=150); plt.show()

**How to read the checks**
* **No target columns in the inputs and no shared fingerprints** → no direct leakage.
* **Shuffled labels fall to about chance** → the models score high only because the Wi-Fi signals really contain the floor information.
* **Stable over 20 splits and good even with few training locations** → not a lucky split and not memorisation.
* **A simple no-ML rule is also very accurate** → floor prediction is an *easy* task: every access point is mounted on one floor and is loudest there.
* **Position is harder**: neighbouring grid points are only ~20 units apart, so a test location always has a close training neighbour. This is why position error looks small in the random test and much larger in the spatial hold-out (Step 7).


## Step 11 – Algorithms used in this project
| Purpose | Algorithm / tool | Where |
|---|---|---|
| Floor classification | **SGD classifier** (logistic regression trained with stochastic gradient descent) | Steps 4, 6, 7 |
| Floor classification | **Random Forest classifier** | Steps 4, 6 |
| Position (X, Y) regression | **K-Nearest Neighbours (KNN) regressor** (k chosen by cross-validation) | Steps 5, 6, 7 |
| Position (X, Y) regression | **Random Forest regressor** | Steps 5, 6 |
| Position (X, Y) regression | **Polynomial Ridge regression** (PCA → degree-2 polynomial features → Ridge) | Step 5 |
| **3D positioning (final system)** | **Two-stage / hierarchical model:** floor classifier → one position regressor per floor | Steps 6, 7 |
| Clustering | **K-Means** (raw fingerprints, and floor-wise BSSID signal strength) | Steps 8, 9 |
| Preprocessing | **StandardScaler**, **PCA** | throughout |
| Evaluation | accuracy, confusion matrix, 2-D / 3-D position error, error CDF, R², silhouette score, elbow method, Adjusted Rand Index, (stratified-group / group) cross-validation | throughout |
| Baselines | majority floor, mean position, oracle floor, no-ML "loudest floor" rule, shuffled labels | Steps 4, 5, 6, 10 |

## Step 12 – Results summary and conclusions
The numbers below are computed from the results above, so they always match the notebook.

In [ ]:
ft, main = results["final_table"], results["main_name"]
m = results["final_rows"][main]
oracle = results["final_table"].iloc[2]
rb = results["robust"]

print("=" * 72)
print("SUMMARY OF RESULTS (all on UNSEEN measurement locations, Wi-Fi signals only)")
print("=" * 72)
print("FINAL 3D SYSTEM:", main)
print(f"  Floor accuracy                 : {m['Floor accuracy']:.1%}   (baseline 'always one floor': {results['floor_baseline']:.1%})")
print(f"  2-D error inside the floor     : mean {m['2D error mean']:.1f} | median {m['2D error median']:.1f} | 90th pct {m['2D error 90th pct']:.1f}")
print(f"  3-D error                      : mean {m['3D error mean']:.1f} | median {m['3D error median']:.1f}  (floor height used: {FLOOR_HEIGHT:.0f})")
print(f"  With the TRUE floor (oracle)   : 2-D mean {oracle['2D error mean']:.1f} | median {oracle['2D error median']:.1f}")
print(f"  Plan size for scale            : about {local_xy[:,0].max():.0f} x {local_xy[:,1].max():.0f} units; mean-position baseline error ~{results['xy_table'].loc['Baseline (mean position)', 'Mean error']:.0f}")
print()
print("STRICTER TESTS (5-fold):")
print(rb.to_string())
print()
print(f"Sanity checks: shuffled-label accuracy {results['shuffled_acc']:.1%} (chance 33.3%), "
      f"no-ML rule {results['rule_acc']:.1%}, average of 20 splits {np.mean(results['repeat_acc']):.1%} (worst {min(results['repeat_acc']):.1%})")
print(f"Clustering vs floors (ARI): raw fingerprints {results['raw_ari']:.3f} | "
      f"floor-wise BSSID signal {results['fs_ari_all']:.3f} (test locations {results['fs_ari_test']:.3f})")

### Conclusions
1. **The old results were not valid.** The old models used PosX/PosY as inputs (and PosY alone identifies the floor), and random splits of raw readings put the same location in train and test.
2. **No leakage was found in the new pipeline, but the dataset has twins.** 48 pairs of neighbouring locations have exactly identical Wi-Fi data. The split keeps twins together. Shuffled labels drop to chance, so the models learn from the signals and not from a hidden answer.
3. **3D multi-floor positioning from Wi-Fi RSSI works.** The two-stage system (floor detection, then position inside that floor) finds the floor very reliably and locates the device inside the floor plan with an error that is small compared with the size of the plan and far below the baseline (see the summary above). The 100 % floor accuracy of Step 4 comes from one particular split; averaged over 20 splits it is a little lower (Step 10), so quote the average.
4. **Floor detection is an easy task.** Every access point is mounted on one floor and is loudest there; even a rule without machine learning is very accurate. Floor errors are rare, but each one is costly for the 3D error.
5. **The position error depends strongly on the test.** Grid points are only ~20 units apart, so in a random-location test every test point has a close training neighbour. When whole strips of the map are held out (Step 7), floor accuracy stays high but the position error grows clearly. Report both.
6. **Clustering.** K-Means on raw fingerprints does **not** recover the floors (ARI ≈ 0), because the signals mainly reflect horizontal position. Clustering the signal strength of floor-wise BSSID groups **does** recover them (ARI close to 1).
7. **Limitations.** Only 273 locations (225 distinct fingerprints) in one building; repeated scans were averaged (removes noise, makes the task easier); the real floor height and the units of PosX/PosY are not documented, so the 3D error uses an assumed floor spacing (`FLOOR_HEIGHT`) and the 2-D error is in file units; no device changes or time variation (RSSI drifts over time and between phones).